In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import pyarrow.feather as feather
import pandas as pd
import numpy as np


In [7]:
table = feather.read_table('../dataset/data_andre.feather', memory_map=True)
df = table.to_pandas()


In [8]:
MAX_PRODUCTS = 100   # set to any number you want


In [9]:
item_ids = df["item_id"].unique()[:MAX_PRODUCTS]
item_id_to_idx = {item_id: i for i, item_id in enumerate(item_ids)}
N = len(item_ids)

# Keep only those products
df = df[df["item_id"].isin(item_ids)].copy()

# Map to node indices
df["node_idx"] = df["item_id"].map(item_id_to_idx)

# Make sure node_idx is integer
df = df.dropna(subset=["node_idx"]).copy()
df["node_idx"] = df["node_idx"].astype(int)

# Make sure date is datetime and normalized consistently
df["date"] = pd.to_datetime(df["date"])


In [ ]:

'''
A time series tensor:

𝑋 ∈ 𝑅 𝑇 × 𝑁 × 𝐹
where:
T = number of time steps (days)

N = number of nodes (items)

F = number of features per node (e.g. value + promos + maybe others)

During training, you turn this into sliding windows:

Input: past history_len steps

Output: next pred_len steps
'''
# Build full date range
full_dates = pd.date_range(start=df["date"].min(), end=df["date"].max(), freq="D")

feature_cols = [
    "value",
    "promo_value_FRPG",
    "promo_value_GAS",
    "promo_value_BOGO",
    "promo_value_DISC",
    "promo_value_CIRC",
    "promo_value_CIRE",
    "promo_value_CLCP",
    "promo_value_LFPE",
]

F_ = len(feature_cols)
T_ = len(full_dates)

# Create time series tensor X with shape (T, N, F)
X = np.zeros((T_, N, F_), dtype=np.float32)

date_to_idx = {date: i for i, date in enumerate(full_dates)}

# Populate X from the DataFrame
for _, row in df.iterrows():
    t = date_to_idx[row["date"].normalize()]  # ensure date-only
    n = row["node_idx"]
    for f_idx, col in enumerate(feature_cols):
        X[t, n, f_idx] = row[col]

X.shape


(761, 100, 9)

# Add temporal features (optional, but MTGNN-style)

In [13]:
from utils import add_time_features_daily
X_all = add_time_features_daily(X, index=full_dates)

print(X_all.shape)  # (T, N, F_total)

(761, 100, 18)


# Building sliding windows

In [ ]:
history_len = 30   # past 30 days
pred_len = 7       # next 7 days

# x_offsets = [-29, -28, ..., 0]
x_offsets = np.arange(-history_len + 1, 1, 1)
# y_offsets = [1, 2, ..., 7]
y_offsets = np.arange(1, pred_len + 1, 1)

In [ ]:
from utils import generate_seq2seq_from_tensor
x, y = generate_seq2seq_from_tensor(X_all, x_offsets, y_offsets)
print("x:", x.shape, "y:", y.shape)